In [80]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Optimization_single_node import *
from Optimization_multi_node import *
import pickle
import copy
import time
import torch
from tqdm.auto import tqdm
from utils_draw import *
from utils import *

In [81]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import matplotlib.patches as mpatches
from matplotlib.ticker import FuncFormatter
import matplotlib as mpl
from matplotlib import font_manager as fm

font_path = '../times.ttf' # 或绝对路径 '/home/me/fonts/times.ttf'
fm.fontManager.addfont(font_path)
prop = fm.FontProperties(fname=font_path)
font_name = prop.get_name()

mpl.rcParams['font.family'] = font_name

In [82]:
class Args:
    def __init__(self):
        # ----- problem / optnet -----
        self.T = 24
        self.base_mva = 100.0
        self.capacity_scale = 4.5
        self.ramp_rate = 0.5
        self.voll = 200.0
        self.vosp = 50.0
        self.M_beta = 1e4
        self.N_scen = 25
        self.pwl_segments = 5

        self.reserve_up_ratio = 0.05
        self.reserve_dn_ratio = 0.02
        self.rt_up_ratio = 3.0
        self.rt_dn_ratio = 0.5

        # ----- training (add) -----
        self.device = "cuda"
        self.epochs = 1
        self.dfl_batch_size = 32
        self.lr = 1e-7
        self.solver = "ECOS"

args=Args()

In [83]:

with open('../Result/VAE/window_pack_s_vae_val.pkl', 'rb') as f:
    window_pack_s_vae_val = pickle.load(f)
with open('../Result/VAE/window_pack_s_vae_train.pkl', 'rb') as f:
    window_pack_s_vae_train = pickle.load(f)
with open('../Result/VAE/window_pack_s_vae_test.pkl', 'rb') as f:
    window_pack_s_vae_test = pickle.load(f) 

with open('../Result/VAE/window_pack_m_vae_test.pkl', 'rb') as f:
    window_pack_m_vae_test = pickle.load(f)
with open('../Result/VAE/window_pack_s_vae_train.pkl', 'rb') as f:
    window_pack_m_vae_train = pickle.load(f)
with open('../Result/VAE/window_pack_m_vae_val.pkl', 'rb') as f:
    window_pack_m_vae_val = pickle.load(f)


In [84]:
with open('../Result/GAN/window_pack_s_gan_test.pkl','rb') as f:
    window_pack_s_gan_test=pickle.load(f)

with open('../Result/GAN/window_pack_s_gan_train.pkl','rb') as f:
    window_pack_s_gan_train=pickle.load(f)

with open('../Result/GAN/window_pack_s_gan_val.pkl','rb') as f:
    window_pack_s_gan_val=pickle.load(f)

with open('../Result/GAN/window_pack_m_gan_test.pkl','rb') as f:
    window_pack_m_gan_test=pickle.load(f)

with open('../Result/GAN/window_pack_m_gan_train.pkl','rb') as f:
    window_pack_m_gan_train=pickle.load(f)

with open('../Result/GAN/window_pack_m_gan_val.pkl','rb') as f:
    window_pack_m_gan_val=pickle.load(f)


In [85]:
with open('../Result/Diffusion/window_pack_m_diff_test.pkl','rb') as f:
    window_pack_m_diff_test=pickle.load(f)
with open('../Result/Diffusion/window_pack_m_diff_val.pkl','rb') as f:
    window_pack_m_diff_val=pickle.load(f)
with open('../Result/Diffusion/window_pack_m_diff_train.pkl', 'rb') as f:
    window_pack_m_diff_train=pickle.load(f)

with open('../Result/Diffusion/window_pack_s_diff_val.pkl', 'rb') as f:
    window_pack_s_diff_val=pickle.load(f)
with open('../Result/Diffusion/window_pack_s_diff_test.pkl', 'rb') as f:
    window_pack_s_diff_test=pickle.load(f)
with open('../Result/Diffusion/window_pack_s_diff_train.pkl', 'rb') as f:
    window_pack_s_diff_train=pickle.load(f)

In [86]:

with open('../Result/Non_parametric/window_pack_non_parametric_val.pkl', 'rb') as f:
    window_pack_non_parametric_val = pickle.load(f)
with open('../Result/Non_parametric/window_pack_non_parametric_train.pkl', 'rb') as f:
    window_pack_non_parametric_train = pickle.load(f)
with open('../Result/Non_parametric/window_pack_non_parametric_test.pkl', 'rb') as f:
    window_pack_non_parametric_test = pickle.load(f)

with open('../Result/Parametric/window_pack_parametric_val.pkl', 'rb') as f:
    window_pack_parametric_val = pickle.load(f)
with open('../Result/Parametric/window_pack_parametric_train.pkl', 'rb') as f:
    window_pack_parametric_train = pickle.load(f)
with open('../Result/Parametric/window_pack_parametric_test.pkl', 'rb') as f:
    window_pack_parametric_test = pickle.load(f)


In [87]:
packs_val = {
     "parametric": window_pack_parametric_val,
     "non_parametric": window_pack_non_parametric_val,
    "diffusion_s": window_pack_s_diff_val,
    "diffusion_m": window_pack_m_diff_val,
     "vae_s": window_pack_s_vae_val,
     "vae_m": window_pack_m_vae_val,
     "gan_s": window_pack_s_gan_val,
     "gan_m": window_pack_m_gan_val,
}

packs_test = {
     "parametric": window_pack_parametric_test,
     "non_parametric": window_pack_non_parametric_test,
     "diffusion_s": window_pack_s_diff_test,
     "diffusion_m": window_pack_m_diff_test,
     "vae_s": window_pack_s_vae_test,
     "vae_m": window_pack_m_vae_test,
     "gan_s": window_pack_s_gan_test,
     "gan_m": window_pack_m_gan_test,
}

packs_train = {
     "parametric": window_pack_parametric_train,
     "non_parametric": window_pack_non_parametric_train,
    "diffusion_s": window_pack_s_diff_train,
    "diffusion_m": window_pack_m_diff_train,
     "vae_s": window_pack_s_vae_train,
     "vae_m": window_pack_m_vae_train,
     "gan_s": window_pack_s_gan_train,
     "gan_m": window_pack_m_gan_train,
}


In [88]:
rows = []
daily_costs = {}

for model_name, pack in packs_val.items():
    print(model_name)

    cost_ideal = Average_cost_Reserve_IDEAL_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )
    cost_det = Average_cost_Reserve_DET_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )
    cost_so = Average_cost_Reserve_SO_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )

    arr_ideal = np.asarray(cost_ideal, dtype=float)
    arr_det = np.asarray(cost_det, dtype=float)
    arr_so = np.asarray(cost_so, dtype=float)

    rows.append({
        "model": model_name,
        "ideal": float(arr_ideal.mean()),
        "deterministic": float(arr_det.mean()),
        "SO": float(arr_so.mean()),
    })

    daily_costs[(model_name, "ideal")] = cost_ideal
    daily_costs[(model_name, "deterministic")] = cost_det
    daily_costs[(model_name, "SO")] = cost_so

df_cost_base_val = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
print(df_cost_base_val)


rows = []
daily_costs = {}
for model_name, pack in packs_test.items():
    print(model_name)

    cost_ideal = Average_cost_Reserve_IDEAL_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )
    cost_det = Average_cost_Reserve_DET_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )
    cost_so = Average_cost_Reserve_SO_MultiNode(
        args=args,
        data=pack,
        n_jobs=None
    )

    arr_ideal = np.asarray(cost_ideal, dtype=float)
    arr_det = np.asarray(cost_det, dtype=float)
    arr_so = np.asarray(cost_so, dtype=float)

    rows.append({
        "model": model_name,
        "ideal": float(arr_ideal.mean()),
        "deterministic": float(arr_det.mean()),
        "SO": float(arr_so.mean()),
    })

    daily_costs[(model_name, "ideal")] = cost_ideal
    daily_costs[(model_name, "deterministic")] = cost_det
    daily_costs[(model_name, "SO")] = cost_so

df_cost_base_test = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
print(df_cost_base_test)

parametric


Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26

In [89]:
set_seed(42)
data_path = "load_data_city_4_2.csv"
target_nodes = [f"4-2-{i}" for i in range(11)]

data = pd.read_csv(data_path)
data["DATETIME"] = pd.to_datetime(data["DATETIME"], errors="coerce")
data_2022 = data[data["DATETIME"].dt.year == 2022].copy()

Lmin, Lmax = system_hourly_load_minmax(
    data_2022, datetime_col="DATETIME", node_cols=target_nodes
)  # (11,24)

Lmin_11 = np.asarray(Lmin, dtype=float)
Lmax_11 = np.asarray(Lmax, dtype=float)

eps_list = [10000,1000,100,10,0]

rows = []
daily_costs = {}  # 可选：保存每个(model, eps)的“每天cost列表”

for model_name, pack in packs_val.items():
    for eps in eps_list:
        print(model_name,eps)
        mode = "ideal" if model_name.startswith("ideal") else None
        cost_list = Average_cost_Reserve_DRO_MultiNode(
            args=args,
            data=pack,
            Lmin_11=Lmin_11,
            Lmax_11=Lmax_11,
            eps=eps,
            mode=mode,
            n_jobs=None,
            return_full=False
        )

        cost_arr = np.asarray(cost_list, dtype=float)
        rows.append({
            "model": model_name,
            "eps": float(eps),
            "mean_cost": float(cost_arr.mean()),
            "std_cost": float(cost_arr.std(ddof=0)),
        })
        daily_costs[(model_name, float(eps))] = cost_list

df_cost_val = pd.DataFrame(rows).sort_values(["model", "eps"]).reset_index(drop=True)
print(df_cost_val)

pivot_mean_val = df_cost_val.pivot(index="model", columns="eps", values="mean_cost")
best_eps_dict = pivot_mean_val.idxmin(axis=1).to_dict()
print(pivot_mean_val)
print(best_eps_dict)

parametric 10000


parametric 1000
parametric 100
parametric 10
parametric 0
non_parametric 10000
non_parametric 1000
non_parametric 100
non_parametric 10
non_parametric 0
diffusion_s 10000
diffusion_s 1000
diffusion_s 100
diffusion_s 10
diffusion_s 0
diffusion_m 10000
diffusion_m 1000
diffusion_m 100
diffusion_m 10
diffusion_m 0
vae_s 10000
vae_s 1000
vae_s 100
vae_s 10
vae_s 0
vae_m 10000
vae_m 1000
vae_m 100
vae_m 10
vae_m 0
gan_s 10000
gan_s 1000
gan_s 100
gan_s 10
gan_s 0
gan_m 10000
gan_m 1000
gan_m 100
gan_m 10
gan_m 0
             model      eps      mean_cost       std_cost
0      diffusion_m      0.0  348704.666689  196671.745415
1      diffusion_m     10.0  348704.666689  196671.745415
2      diffusion_m    100.0  348052.344000  195722.410225
3      diffusion_m   1000.0  348112.947598  119546.511905
4      diffusion_m  10000.0  348520.228266  120330.301602
5      diffusion_s      0.0  362748.906575  206507.831915
6      diffusion_s     10.0  362748.906575  206507.831915
7      diffusion_s    1

In [93]:
best_eps_dict = pivot_mean_val.idxmin(axis=1).to_dict()
print(pivot_mean_val)
print(best_eps_dict)

eps                   0.0            10.0           100.0          1000.0   \
model                                                                        
diffusion_m     348704.666689  348704.666689  348052.344000  348112.947598   
diffusion_s     362748.906575  362748.906575  361877.288015  351629.717108   
gan_m           345964.713473  345964.713473  345607.282102  345922.729284   
gan_s           370247.588481  370247.588481  369792.657373  353762.206167   
non_parametric  372620.970096  372620.970096  370256.341390  356576.562233   
parametric      376027.793005  376027.793005  372194.556488  361041.947735   
vae_m           341524.266866  341524.266866  343198.639690  347272.022276   
vae_s           378092.967463  378092.967463  374956.172837  354756.805440   

eps                   10000.0  
model                          
diffusion_m     348520.228266  
diffusion_s     352290.904905  
gan_m           346631.956957  
gan_s           354516.401448  
non_parametric  357682.6906

In [94]:
# 先把 pivot_mean_val 从 index 变回普通列
df_dro = pivot_mean_val.reset_index()

# 再和 base 表按 model 合并
df_merged_val = pd.merge(
    df_cost_base_val,
    df_dro,
    on="model",
    how="inner"
)

df_merged_val

,model,ideal,deterministic,SO,0.0,10.0,100.0,1000.0,10000.0
0,diffusion_m,291877.185166,399591.631707,348704.666689,348704.666689,348704.666689,348052.344000,348112.947598,348520.228266
1,diffusion_s,291877.185166,400963.715933,362748.906575,362748.906575,362748.906575,361877.288015,351629.717108,352290.904905
2,gan_m,291877.185166,385615.589875,345964.713473,345964.713473,345964.713473,345607.282102,345922.729284,346631.956957
3,gan_s,291877.185166,395896.045853,370247.588481,370247.588481,370247.588481,369792.657373,353762.206167,354516.401448
4,non_parametric,291877.185497,407073.974938,372620.970096,372620.970096,372620.970096,370256.341390,356576.562233,357682.690618
5,parametric,291877.185166,429368.836146,376027.793005,376027.793005,376027.793005,372194.556488,361041.947735,361997.040240
6,vae_m,291877.185166,395247.152240,341524.266866,341524.266866,341524.266866,343198.639690,347272.022276,347607.748091
7,vae_s,291877.185166,406676.828759,378092.967463,378092.967463,378092.967463,374956.172837,354756.805440,355140.972587


In [99]:
rows = []

for model_name, pack in packs_test.items():
    eps = best_eps_dict[model_name]
    print(model_name, eps)

    mode = "ideal" if model_name.startswith("ideal") else None
    cost_list = Average_cost_Reserve_DRO_MultiNode(
        args=args,
        data=pack,
        Lmin_11=Lmin_11,
        Lmax_11=Lmax_11,
        eps=eps,
        mode=mode,
        n_jobs=None,
        return_full=False
    )

    cost_arr = np.asarray(cost_list, dtype=float)

    rows.append({
        "model": model_name,
        "eps": float(eps),
        "cost": float(cost_arr.mean()),
    })

df_cost_test = pd.DataFrame(rows).sort_values("model").reset_index(drop=True)
print(df_cost_test)

parametric 1000.0
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - expires 2026-08-26
Set parameter Username
Academic license - for non-commercial use only - 

In [100]:
df_best_dro_test = df_cost_test.reset_index()
df_merge_test = pd.merge(
    df_cost_base_test,
    df_best_dro_test[["model", "eps", "cost"]].rename(columns={"cost": "DRO"}),
    on="model",
    how="inner"
)

df_merge_test

,model,ideal,deterministic,SO,eps,DRO
0,diffusion_m,311305.178760,415252.874896,360599.743893,100.0,358483.294405
1,diffusion_s,311305.178760,403170.700203,368118.470743,1000.0,369387.225533
2,gan_m,311305.178760,398961.795943,357506.460791,100.0,354311.244096
3,gan_s,311305.178760,397266.383726,372892.108926,1000.0,371751.487403
4,non_parametric,311305.179032,424276.514846,388708.766963,1000.0,380043.723492
5,parametric,311305.178760,424386.572664,378648.915193,1000.0,378777.889412
6,vae_m,311305.178760,407832.515050,352981.458874,0.0,352981.458874
7,vae_s,311305.178760,414894.427414,387975.994217,1000.0,374213.814690


In [102]:
df_merged_val.to_csv('../Result/cost on validation with different eps.csv')

In [101]:
df_merge_test.to_csv('../Result/eps_search.csv')